[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jman4162/sensortwin-transformer-agent/blob/master/notebooks/04_colab_standard_comparison.ipynb)

# SensorTwin — the headline comparison at `colab_standard`

Runs `scripts/compare_models.py` — the same one command as `make headline` — over the full slate
(logreg / random_forest / xgboost / cnn / lstm / **SensorPatchTST**) at `colab_standard` (20k
samples), 5 seeds. All deep models share one training recipe family (`configs/models/*.yaml`), so
the comparison measures architecture, not tuning budget. Output: mean ± sample std per model, a
paired t-interval + t-test + Cohen's d for transformer-vs-best-baseline, and Holm-corrected
per-class deltas, written to `reports/experiment_summaries/headline_comparison.{md,json}`.

Budget: roughly 1–1.5 h per seed on a T4; drop `SEEDS` or use `MODE = "quick_demo"` for a dry run.

In [ ]:
# Opened from the Colab badge? Only the notebook is present — clone the public repo, then install.
import os

if not os.path.exists("sensortwin"):
    !git clone https://github.com/jman4162/sensortwin-transformer-agent.git
    %cd sensortwin-transformer-agent
%pip install -q -e ".[ml]"

In [ ]:
import torch

print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — 20k x 5 seeds is slow on CPU; consider MODE='quick_demo'.")

In [ ]:
# --- run configuration (edit here) ---
MODE = "colab_standard"      # 20k samples; quick_demo (2k) for a fast dry run
SEEDS = [0, 1, 2, 3, 4]
EPOCHS = 30
OUT = "reports/experiment_summaries"

In [ ]:
# --- the headline run: one command, committable artifacts ---
from scripts.compare_models import main as compare_models

compare_models(
    ["--mode", MODE, "--seeds", *[str(s) for s in SEEDS], "--epochs", str(EPOCHS), "--out", OUT]
)

In [ ]:
# --- render the report inline ---
from pathlib import Path

from IPython.display import Markdown, display

display(Markdown(Path(f"{OUT}/headline_comparison.md").read_text()))

In [ ]:
# --- download the artifacts (Colab is ephemeral) ---
try:
    from google.colab import files

    files.download(f"{OUT}/headline_comparison.md")
    files.download(f"{OUT}/headline_comparison.json")
except Exception as e:
    print("Not in Colab or download unavailable:", e)
    print(f"Artifacts are in {OUT}/headline_comparison.{{md,json}}")

## Reading the result

- The **per-seed columns** are the raw evidence; everything else is derived from them.
- The transformer-vs-best-baseline delta is only claimable if the paired t-test and t-interval
  agree, and per-class deltas only where they survive Holm correction (m = 10).
- Commit `headline_comparison.{md,json}` so every number in the README traces to this run.
- Follow-ups at the same scale: `05_calibration_colab`, `06_label_efficiency_colab`,
  `07_robustness_colab`.